# Part II — Python Coding Problems
## Masked Self-Attention and Vision Transformer (ViT) Patch Embedding



# Coding Problem 1. Implement Masked Self-Attention

We implement scaled dot-product attention with:
- **Causal masking:** a token can attend only to itself and earlier tokens.
- **Padding masking:** padded key positions cannot receive attention; padded query outputs are set to zero.

For this example, `Q`, `K`, and `V` each contain four token vectors.

## 1(a). Scaled dot-product attention without masking

The function calculates `Q @ K.T`, divides by `sqrt(d_k)`, applies softmax row-wise, and multiplies the attention weights by `V`.

In [1]:
import numpy as np

def softmax(x, axis=-1):
    # Numerically stable softmax
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    # Step 1: Calculate attention scores
    scores = np.matmul(Q, K.T)

    # Step 2: Scale by the square root of key dimension
    d_k = K.shape[-1]
    scaled_scores = scores / np.sqrt(d_k)

    # Step 3: Convert scores into attention weights
    attention_weights = softmax(scaled_scores, axis=-1)

    # Step 4: Calculate weighted sum of values
    output = np.matmul(attention_weights, V)

    return attention_weights, output


# Four example token vectors (each has dimension 2)
Q = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.5, 0.5]
])

K = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.5, 0.5]
])

V = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [2.0, 2.0]
])

weights_unmasked, output_unmasked = scaled_dot_product_attention(Q, K, V)

print("Attention weights BEFORE masking:")
print(np.round(weights_unmasked, 3))

print("\nOutput BEFORE masking:")
print(np.round(output_unmasked, 3))

Attention weights BEFORE masking:
[[0.313 0.154 0.313 0.22 ]
 [0.154 0.313 0.313 0.22 ]
 [0.199 0.199 0.403 0.199]
 [0.226 0.226 0.322 0.226]]

Output BEFORE masking:
[[1.065 0.907]
 [0.907 1.065]
 [1.    1.   ]
 [1.    1.   ]]


## 1(b). Add causal masking

The causal mask is a lower-triangular matrix. A value of `1` means attention is allowed; `0` means it is blocked. We apply the mask to the scores **before softmax**.

In [2]:
def scaled_dot_product_attention(Q, K, V, causal_mask=False,
                                  padding_mask=None):
    # Step 1: Calculate and scale attention scores
    scores = np.matmul(Q, K.T)
    d_k = K.shape[-1]
    scaled_scores = scores / np.sqrt(d_k)

    # Start with every query-key position allowed
    allowed = np.ones_like(scaled_scores, dtype=bool)

    # Step 2: Apply causal mask, if requested
    if causal_mask:
        seq_len = Q.shape[0]
        causal = np.tril(np.ones((seq_len, seq_len), dtype=bool))
        allowed = allowed & causal

    # Step 3: Apply padding mask to key positions, if provided
    if padding_mask is not None:
        valid_keys = np.asarray(padding_mask).astype(bool)
        allowed = allowed & valid_keys[np.newaxis, :]

    # Step 4: Mask disallowed scores before softmax
    masked_scores = np.where(allowed, scaled_scores, -1e9)
    attention_weights = softmax(masked_scores, axis=-1)

    # Make disallowed attention weights exactly zero
    attention_weights = attention_weights * allowed

    # Renormalize each row (handles rows with at least one allowed key)
    row_sums = attention_weights.sum(axis=-1, keepdims=True)
    attention_weights = np.divide(
        attention_weights,
        row_sums,
        out=np.zeros_like(attention_weights),
        where=row_sums != 0
    )

    # Step 5: Weighted sum of values
    output = np.matmul(attention_weights, V)

    # Padded query positions must have zero context output
    if padding_mask is not None:
        valid_queries = np.asarray(padding_mask).astype(bool)
        output[~valid_queries] = 0
        attention_weights[~valid_queries] = 0

    return attention_weights, output


# The required 4-token causal mask
causal_mask = np.tril(np.ones((4, 4), dtype=int))
print("Causal mask (1 = allowed, 0 = blocked):")
print(causal_mask)

causal_weights, causal_output = scaled_dot_product_attention(
    Q, K, V, causal_mask=True
)

print("\nAttention weights AFTER causal masking:")
print(np.round(causal_weights, 3))

print("\nOutput AFTER causal masking:")
print(np.round(causal_output, 3))

Causal mask (1 = allowed, 0 = blocked):
[[1 0 0 0]
 [1 1 0 0]
 [1 1 1 0]
 [1 1 1 1]]

Attention weights AFTER causal masking:
[[1.    0.    0.    0.   ]
 [0.33  0.67  0.    0.   ]
 [0.248 0.248 0.503 0.   ]
 [0.226 0.226 0.322 0.226]]

Output AFTER causal masking:
[[1.    0.   ]
 [0.33  0.67 ]
 [0.752 0.752]
 [1.    1.   ]]


## 1(c). Add padding masking and test both masks

The required padding mask is `[1, 1, 1, 0]`. The fourth token is padding:
- Its key column receives zero attention weight.
- Its query row has zero attention weights.
- Its context output is set to zero.

We test padding masking alone and causal + padding masking together.

In [3]:
padding_mask = np.array([1, 1, 1, 0])

# Padding masking alone
padding_weights, padding_output = scaled_dot_product_attention(
    Q, K, V, padding_mask=padding_mask
)

print("Padding mask:", padding_mask)
print("\nAttention weights AFTER padding masking:")
print(np.round(padding_weights, 3))

print("\nOutput AFTER padding masking:")
print(np.round(padding_output, 3))

# Apply causal and padding masks together
combined_weights, combined_output = scaled_dot_product_attention(
    Q, K, V,
    causal_mask=True,
    padding_mask=padding_mask
)

print("\nAttention weights AFTER causal + padding masking:")
print(np.round(combined_weights, 3))

print("\nOutput AFTER causal + padding masking:")
print(np.round(combined_output, 3))

# Verify the required conditions
assert np.allclose(combined_weights[:, 3], 0), "Padding key column is not zero."
assert np.allclose(combined_weights[3, :], 0), "Padding query row is not zero."
assert np.allclose(combined_output[3], 0), "Padding query output is not zero."

print("\nVerification passed:")
print("- Padded key positions receive zero attention weight.")
print("- Padded query positions have zero attention weights and zero output.")
print("- Causal masking blocks all future-token positions.")

Padding mask: [1 1 1 0]

Attention weights AFTER padding masking:
[[0.401 0.198 0.401 0.   ]
 [0.198 0.401 0.401 0.   ]
 [0.248 0.248 0.503 0.   ]
 [0.    0.    0.    0.   ]]

Output AFTER padding masking:
[[0.802 0.599]
 [0.599 0.802]
 [0.752 0.752]
 [0.    0.   ]]

Attention weights AFTER causal + padding masking:
[[1.    0.    0.    0.   ]
 [0.33  0.67  0.    0.   ]
 [0.248 0.248 0.503 0.   ]
 [0.    0.    0.    0.   ]]

Output AFTER causal + padding masking:
[[1.    0.   ]
 [0.33  0.67 ]
 [0.752 0.752]
 [0.    0.   ]]

Verification passed:
- Padded key positions receive zero attention weight.
- Padded query positions have zero attention weights and zero output.
- Causal masking blocks all future-token positions.


### Coding Problem 1 — Explanation

**Causal masking** prevents a token from attending to future tokens. It is used in autoregressive generation so that a token is predicted using only itself and earlier tokens.

**Padding masking** prevents the model from attending to padding tokens that were added to make sequences the same length. In this implementation, padded key positions receive zero attention weight, and padded query positions produce zero attention weights and zero context output.

**Difference:** Causal masking is based on token position; padding masking is based on whether a token is real or padding.

# Coding Problem 2. Implement Vision Transformer Patch Embedding

Configuration:
- Image size: `32 × 32`
- Channels: `3` (RGB)
- Patch size: `8 × 8`
- Embedding dimension: `64`
- Batch size: `2`

Expected number of patches per image: `(32 / 8) × (32 / 8) = 16`.

## 2(a). Image patch extraction

Input images have shape `(B, C, H, W)`. We reshape and permute the image tensor so each non-overlapping patch becomes one flattened vector. The output shape is `(B, N, P × P × C)`.


In [4]:
import torch
import torch.nn as nn

def extract_patches(images, patch_size):
    # images shape: (B, C, H, W)
    B, C, H, W = images.shape
    P = patch_size

    assert H % P == 0 and W % P == 0, "Image dimensions must be divisible by patch size."

    # Split height and width into patch grids
    patches = images.reshape(
        B, C, H // P, P, W // P, P
    )

    # Move patch-grid dimensions before patch contents
    patches = patches.permute(0, 2, 4, 1, 3, 5)

    # Flatten each patch: P * P * C values
    patches = patches.reshape(B, (H // P) * (W // P), C * P * P)

    return patches


# Generate two random RGB images
images = torch.randn(2, 3, 32, 32)

patch_size = 8
patches = extract_patches(images, patch_size)

print("Input image shape:", images.shape)
print("Extracted patches shape:", patches.shape)
print("Number of patches per image:", patches.shape[1])
print("Flattened patch dimension:", patches.shape[2])

Input image shape: torch.Size([2, 3, 32, 32])
Extracted patches shape: torch.Size([2, 16, 192])
Number of patches per image: 16
Flattened patch dimension: 192


## 2(b). Patch embedding and positional encoding

The `ViTPatchEmbedding` module:
1. Projects every flattened patch to 64 dimensions using a learnable linear layer.
2. Creates a learnable `[CLS]` token.
3. Adds the `[CLS]` token at the beginning of the sequence.
4. Adds learnable positional embeddings to all tokens.

In [11]:
import torch
import torch.nn as nn

# Step 1: Extract image patches
def extract_patches(images, patch_size):
    B, C, H, W = images.shape

    patches = images.unfold(2, patch_size, patch_size)
    patches = patches.unfold(3, patch_size, patch_size)

    # Rearrange patches
    patches = patches.permute(0, 2, 3, 1, 4, 5)

    # Flatten each patch
    patches = patches.reshape(B, -1, C * patch_size * patch_size)

    return patches


# Step 2: Create ViT Patch Embedding class
class ViTPatchEmbedding(nn.Module):

    def __init__(self, image_size=32, patch_size=8,
                 in_channels=3, embedding_dim=64):

        super().__init__()

        self.image_size = image_size
        self.patch_size = patch_size

        self.num_patches = (image_size // patch_size) ** 2

        # Project flattened patches into 64 dimensions
        patch_dim = in_channels * patch_size * patch_size
        self.projection = nn.Linear(patch_dim, embedding_dim)

        # Learnable CLS token
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, embedding_dim)
        )

        # Positional embeddings
        self.position_embedding = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, embedding_dim)
        )

    def forward(self, images):

        # Extract patches
        patches = extract_patches(images, self.patch_size)

        # Convert patches into embeddings
        patch_embeddings = self.projection(patches)

        # Add CLS token
        B = images.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1)

        tokens = torch.cat((cls_tokens, patch_embeddings), dim=1)

        # Add positional embeddings
        tokens = tokens + self.position_embedding

        return tokens


# Step 3: Test the model
images = torch.randn(2, 3, 32, 32)

model = ViTPatchEmbedding()

output = model(images)

print("Input Image Shape:", images.shape)
print("Final Output Shape:", output.shape)

# Step 4: Verify the output
assert output.shape == (2, 17, 64)

print("Output shape is correct!")

Input Image Shape: torch.Size([2, 3, 32, 32])
Final Output Shape: torch.Size([2, 17, 64])
Output shape is correct!


## 2(c). Test the ViT patch embedding module

Print the number of patches, flattened patch dimension, patch embedding shape before adding `[CLS]`, and final output shape.

In [6]:
# Model configuration
image_size = 32
patch_size = 8
embedding_dim = 64
batch_size = 2
channels = 3

# Create random RGB images
images = torch.randn(batch_size, channels, image_size, image_size)

# Extract patches to inspect their dimensions
patches = extract_patches(images, patch_size)

# Create and run the ViT patch embedding module
vit_embedding = ViTPatchEmbedding(
    image_size=image_size,
    patch_size=patch_size,
    in_channels=channels,
    embedding_dim=embedding_dim
)

patch_embeddings, final_embeddings = vit_embedding(images)

print("Input image shape:", images.shape)
print("Number of patches per image:", patches.shape[1])
print("Flattened patch dimension:", patches.shape[2])
print("Patch embedding shape BEFORE adding CLS:", patch_embeddings.shape)
print("Final embedding shape AFTER CLS and positional embeddings:", final_embeddings.shape)

# Verify expected dimensions
assert patches.shape == (2, 16, 192)
assert patch_embeddings.shape == (2, 16, 64)
assert final_embeddings.shape == (2, 17, 64)

print("\nVerification passed: final output shape is (2, 17, 64).")

Input image shape: torch.Size([2, 3, 32, 32])
Number of patches per image: 16
Flattened patch dimension: 192
Patch embedding shape BEFORE adding CLS: torch.Size([2, 16, 64])
Final embedding shape AFTER CLS and positional embeddings: torch.Size([2, 17, 64])

Verification passed: final output shape is (2, 17, 64).


### Expected dimensions

| Item | Expected shape/value |
|---|---:|
| Input images | `(2, 3, 32, 32)` |
| Patches per image | `16` |
| Flattened patch dimension | `8 × 8 × 3 = 192` |
| Extracted patches | `(2, 16, 192)` |
| Patch embeddings before CLS | `(2, 16, 64)` |
| Final sequence after CLS + positions | `(2, 17, 64)` |

### Coding Problem 2 — Explanation

**Positional embeddings:** Self-attention does not automatically know the spatial order of image patches. Positional embeddings provide information about each patch's position in the image.

**Classification token ([CLS]):** The learnable `[CLS]` token is added at the beginning of the patch sequence. Through self-attention, it collects information from the image patches. Its final representation can be passed to a classification layer to predict the image class.

# Final Revision

**Problem 1 — Masked Self-Attention**
- Attention scores: `Q @ K.T`
- Scaling: divide by `sqrt(d_k)`
- Causal mask: lower-triangular matrix
- Padding mask: blocks invalid key positions
- Softmax is applied after masking
- Padded query output is set to zero

**Problem 2 — ViT Patch Embedding**
- Extract patches with `reshape()` and `permute()`
- Flattened patch size: `P × P × C`
- Use `nn.Linear()` to project patches
- Use `nn.Parameter()` for learnable `[CLS]` and positional embeddings
- Concatenate `[CLS]` before adding positional embeddings

**Key expected final ViT shape:** `(2, 17, 64)`